# 05 — CLI & Pipelines


Démos `convert-csv`, `dedup` et squelette de pipeline scriptable (sans dépendre du binaire CLI).


In [ ]:

from pathlib import Path
import pandas as pd, numpy as np
try:
    from optech_raman_ai.data.convert_csv_to_parquet import convert_folder as convert_csv_folder
    HAVE_CONVERT=True
except Exception as e:
    HAVE_CONVERT=False; print("[WARN] convert_csv_folder non dispo:", e)
try:
    from optech_raman_ai.utils.deduplicate import deduplicate_folder
    HAVE_DEDUP=True
except Exception as e:
    HAVE_DEDUP=False; print("[WARN] deduplicate_folder non dispo:", e)

raw = Path("raw_cli_demo"); raw.mkdir(exist_ok=True)
for i in range(3):
    x = np.linspace(100, 2000, 512)
    y = np.sin(x/80.0) + 0.01*np.random.randn(512)
    pd.DataFrame({"sample_id":[f"s{i}"]*512,"wavenumber_cm1":x,"intensity":y}).to_csv(raw/f"s{i}.csv", index=False)

out = Path("proc_cli_demo"); out.mkdir(exist_ok=True)
if HAVE_CONVERT:
    convert_csv_folder(raw, out)
else:
    for f in raw.glob("*.csv"):
        pd.read_csv(f).to_parquet(out/(f.stem+".parquet"), index=False)

if HAVE_DEDUP:
    deduplicate_folder(out, algo="sha1")
list(out.glob("*.parquet"))[:5]
